In [0]:
%pip install databricks-feature-engineering -q

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
from databricks.feature_engineering import FeatureEngineeringClient
from databricks.feature_engineering import FeatureLookup
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 1. Configuración de Unity Catalog y Feature Store
fe = FeatureEngineeringClient()
CATALOG = "workspace"
SCHEMA = "gold"
FEATURE_TABLE = f"{CATALOG}.{SCHEMA}.store_performance_features"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.walmart_sales_predictor"

# 2. Definir los "Lookups" del Feature Store
# Esto le dice a Databricks: "Ve al Feature Store y tráeme estas columnas automáticamente"
feature_lookups = [
    FeatureLookup(
      table_name=FEATURE_TABLE,
      lookup_key="store_id",
      timestamp_lookup_key="sales_date",
      feature_names=["feat_rolling_transactions", "feat_rolling_revenue", "feat_items_sold"]
    )
]

# 3. Crear el Training Set
# Nota: 'raw_data' sería una tabla pequeña con store_id y el 'target' (lo que quieres predecir)
raw_data = spark.table(f"{CATALOG}.gold.regional_sales_performance").select("store_id", "sales_date", "total_revenue")

training_set = fe.create_training_set(
    df=raw_data,
    feature_lookups=feature_lookups,
    label="total_revenue", # En este ejemplo, intentamos predecir el revenue actual basado en features históricas
    exclude_columns=["sales_date"]
)

training_df = training_set.load_df().toPandas().dropna()
X = training_df.drop(["total_revenue", "store_id"], axis=1)
y = training_df["total_revenue"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# 4. Entrenamiento con MLflow (Tracking)
with mlflow.start_run(run_name="Walmart_Sales_Forecasting") as run:
    # Definir y entrenar el modelo
    rf = RandomForestRegressor(n_estimators=100)
    rf.fit(X_train, y_train)
    
    # Calcular métricas
    predictions = rf.predict(X_test)
    rmse = mean_squared_error(y_test, predictions) ** 0.5
    mlflow.log_metric("rmse", rmse)
    
    # 5. Registrar el modelo en Unity Catalog usando Feature Engineering
    # Esto vincula el modelo permanentemente con sus features
    fe.log_model(
        model=rf,
        artifact_path="sales_model",
        flavor=mlflow.sklearn,
        training_set=training_set,
        registered_model_name=MODEL_NAME
    )
    
    print(f"Modelo entrenado con RMSE: {rmse}. Registrado en Unity Catalog como: {MODEL_NAME}")